# 02 — Feature analysis (G1–G5)

The prefix features the tabular models see (`features.groups` in `config.yaml`), examined **on the training split only**
(validation and test are touched once, in section 8, to measure drift, never to choose
anything).

| Section | Question | Feeds |
|---|---|---|
| 1 | What are the features and how are they encoded? | method section |
| 2 | Are the tables sound (no gaps, no leakage, K respected)? | validity |
| 3 | How do buyers and non-buyers differ, feature by feature? | RQ3, H5 |
| 4 | Which categorical values carry signal? | G2 / G4 / G5 |
| 5 | How much can one feature predict on its own? | RQ3 baseline intuition |
| 6 | Which features are redundant — and why H3 is hard by construction | RQ3 (H3 and the variant) |
| 7 | Why the log transform: ratio and price tails | preprocessing |
| 8 | Do feature distributions shift from train to test? | chronological split, threats |
| 9 | What the GRU sees: event types by position | RQ1 (H1c) |
| 10 | Key numbers | report |

**Run order.** Steps 1–2 first (`scripts/01_prepare_data.py`, `scripts/02_build_features.py`);
synthetic data is built on the fly. Choose the data with `NB_MODE=dev jupyter lab`.
Figures go to `results/figures/feat_*_<mode>.png`, key numbers to
`results/tables/feature_analysis_<mode>.json`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import average_precision_score, roc_auc_score

from src.config import feature_list, processed_dir_for_k, resolve_path
from src.data.sessionize import ANALYSIS_ONLY_COLUMNS, TARGET
from src.features.build_features import load_feature_table
from src.features.preprocessing import prepare_splits
from src.utils.notebook import (EVENT_COLOURS, INK_MUTED, PALETTE, TARGET_COLOURS, ensure_features,
                                notebook_config, save_figure, use_style)

cfg = notebook_config()
ensure_features(cfg)
use_style()
pl.Config.set_tbl_rows(30)
pl.Config.set_thousands_separator(",")

MODE = cfg["data"]["mode"]
KS = sorted(cfg["sessionize"]["prefix_k"])
K0 = cfg["sessionize"]["default_k"]
GROUPS = cfg["features"]["groups"]
GROUP_OF = {f: g for g, fs in GROUPS.items() for f in fs}
GROUP_COLOURS = {g: PALETTE[i] for i, g in enumerate(GROUPS)}       # fixed order G1..G5
FEATURES = feature_list(cfg)
CATEGORICAL = cfg["features"]["categorical"]
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
LOG1P = set(cfg["preprocessing"]["log1p_features"])
SEED = cfg["project"]["seeds"][0]
COUNTS = [c for c in ("view_count", "cart_count", "remove_count") if c in FEATURES]   # funnel counts in use
RATIOS = list(GROUPS.get("G3_behavioural_ratios", []))

tables = {k: load_feature_table(cfg, k) for k in KS}
train = {k: t.filter(pl.col("split") == "train") for k, t in tables.items()}
summary = {"mode": MODE, "n_features": len(FEATURES)}


def model_scale(df: pl.DataFrame, f: str) -> np.ndarray:
    # The scale linear / neural models see (log1p for heavy-tailed features).
    x = df[f].cast(pl.Float64).to_numpy()
    return np.log1p(np.clip(x, 0, None)) if f in LOG1P else x


def group_legend(ax, **kw):
    handles = [Line2D([], [], marker="s", linestyle="", markersize=9, color=c, label=g)
               for g, c in GROUP_COLOURS.items()]
    ax.legend(handles=handles, **kw)


print(f"mode={MODE}  K={KS}  default K={K0}  features={len(FEATURES)} "
      f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")

## 1. Feature catalogue

In [ ]:
DESCRIPTIONS = {
    "unique_products_count": "distinct products in the prefix",
    "session_duration_sec": "time from event 1 to event K",
    "max_gap_sec": "longest pause between consecutive prefix events",
    "view_count": "views in the prefix (= K − cart − remove)",
    "cart_count": "add-to-cart events",
    "remove_count": "remove-from-cart events",
    "has_cart": "at least one cart event (0/1)",
    "first_cart_position": "position of the first cart, K+1 if none",
    "last_event_type": "type of event K",
    "cart_to_view_ratio": "cart_count / (view_count + ε)",
    "remove_to_cart_ratio": "remove_count / (cart_count + ε)",
    "unique_categories_count": "distinct category ids",
    "unique_brands_count": "distinct brands",
    "main_category": "top-level category of the most-viewed product",
    "price_mean": "mean price of prefix events",
    "price_max": "highest price seen",
    "price_sum": "sum of prices seen",
    "price_std": "price spread (0 for a single price)",
    "hour_of_day": "hour of event K (UTC)",
    "day_of_week": "weekday of event K (Mon = 1)",
}
profiles = cfg["preprocessing"]["model_profiles"]
catalogue = pl.DataFrame([{
    "group": GROUP_OF[f], "feature": f,
    "type": "categorical" if f in CATEGORICAL else "numeric",
    "log1p (LR/MLP/TabNet)": f in LOG1P,
    "description": DESCRIPTIONS.get(f, ""),
} for f in FEATURES])
display(catalogue)
print("Encoding per model:", profiles)

## 2. Integrity checks

Every check must pass before any modelling. Because purchases inside the prefix were removed
by the leakage rules, the funnel counts add up to K (exactly when `remove_count` is a feature;
at most K otherwise — the REES46 multi-category data has no remove events, so it is left out).

In [ ]:
rows = []
for k, t in tables.items():
    rows.append({
        "K": k,
        "sessions": t.height,
        "unique_sessions": t["user_session"].n_unique() == t.height,
        "missing_values": int(sum(t.select(FEATURES).null_count().row(0))),
        "analysis_only_cols": sorted(set(t.columns) & (set(ANALYSIS_ONLY_COLUMNS) - {TARGET})),
        "counts_fit_K": bool((t.select(pl.sum_horizontal(COUNTS)).to_series() == k).all() if "remove_count" in COUNTS
                             else (t.select(pl.sum_horizontal(COUNTS)).to_series() <= k).all()),
        "first_cart_in_1..K+1": bool(t["first_cart_position"].is_between(1, k + 1).all()),
        "has_cart_matches": bool(((t["cart_count"] > 0).cast(pl.Int8) == t["has_cart"].cast(pl.Int8)).all()),
    })
checks = pl.DataFrame(rows)
display(checks)
ok = all(r["unique_sessions"] and r["missing_values"] == 0 and not r["analysis_only_cols"] and r["counts_fit_K"]
         and r["first_cart_in_1..K+1"] and r["has_cart_matches"] for r in rows)
print("ALL CHECKS PASSED" if ok else "CHECK FAILURES — fix before modelling")
summary["integrity_ok"] = ok

sizes = pl.concat([
    t.group_by("split").agg(sessions=pl.len(), positive_rate=pl.col(TARGET).mean()).with_columns(K=pl.lit(k))
    for k, t in tables.items()
]).pivot(on="split", index="K", values=["sessions", "positive_rate"]).sort("K")
display(sizes)

## 3. Buyers vs non-buyers

**Standardised mean difference** (SMD) = (mean of buyers − mean of non-buyers) ÷ pooled SD,
on the scale the linear and neural models see. |SMD| ≥ 0.2 is small, ≥ 0.5 medium,
≥ 0.8 large. Positive = higher for buyers. One dot per K.

In [ ]:
def smd_table(df: pl.DataFrame) -> pl.DataFrame:
    y = df[TARGET].to_numpy() == 1
    out = []
    for f in NUMERIC:
        x = model_scale(df, f)
        a, b = x[y], x[~y]
        pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if len(a) > 1 and len(b) > 1 else 0.0
        out.append({"feature": f, "group": GROUP_OF[f], "mean_buyers": a.mean() if len(a) else np.nan,
                    "mean_non_buyers": b.mean() if len(b) else np.nan,
                    "smd": (a.mean() - b.mean()) / pooled if pooled > 0 else 0.0})
    return pl.DataFrame(out)

smd = pl.concat([smd_table(train[k]).with_columns(K=pl.lit(k)) for k in KS])
smd_k0 = smd.filter(pl.col("K") == K0).sort(pl.col("smd").abs(), descending=True)
display(smd_k0)
summary["top_smd"] = smd_k0.head(5).select("feature", "smd").to_dicts()

In [ ]:
order = smd_k0.sort(pl.col("smd").abs())["feature"].to_list()
markers = ["o", "s", "D", "^", "v"]
fig, ax = plt.subplots(figsize=(8, 0.34 * len(order) + 1.2))
ax.axvline(0, color=INK_MUTED, linewidth=1)
for v in (-0.8, -0.5, -0.2, 0.2, 0.5, 0.8):
    ax.axvline(v, color="#e4e3df", linewidth=1, linestyle=":")
for i, k in enumerate(KS):
    d = dict(zip(*smd.filter(pl.col("K") == k).select("feature", "smd").to_dict(as_series=False).values()))
    ax.scatter([d[f] for f in order], range(len(order)), s=42, marker=markers[i % len(markers)],
               color=[GROUP_COLOURS[GROUP_OF[f]] for f in order], edgecolor="white", linewidth=1.2,
               zorder=3, label=f"K={k}")
ax.set_yticks(range(len(order)), order)
ax.grid(axis="y", visible=False)
ax.set_xlabel("standardised mean difference (buyers − non-buyers)")
ax.set_title(f"Buyer vs non-buyer difference per feature (train split; sorted by K={K0})")
leg1 = ax.legend(handles=[Line2D([], [], marker=markers[i % len(markers)], linestyle="", color=INK_MUTED,
                                 label=f"K={k}") for i, k in enumerate(KS)], loc="lower left", bbox_to_anchor=(1.02, 0), title="marker")
ax.add_artist(leg1)
group_legend(ax, loc="upper left", bbox_to_anchor=(1.02, 1), title="colour")
save_figure(fig, cfg, "feat_smd")
plt.show()

**Distributions behind the biggest differences** (default K, train split, model scale).

In [ ]:
top = smd_k0.head(8)["feature"].to_list()
fig, axes = plt.subplots(2, 4, figsize=(13, 5.4))
t = train[K0]
y = t[TARGET].to_numpy() == 1
for ax, f in zip(axes.flat, top):
    x = model_scale(t, f)
    discrete = np.unique(x).size <= 12
    bins = np.arange(x.min() - 0.5, x.max() + 1.5) if discrete and np.allclose(x, np.round(x)) else 30
    for cls, label in ((0, "non-buyer"), (1, "buyer")):
        vals = x[y == bool(cls)]
        if vals.size:
            ax.hist(vals, bins=bins, density=True, histtype="step", linewidth=2, color=TARGET_COLOURS[cls], label=label)
    ax.set_title(("log1p " if f in LOG1P else "") + f, fontsize=10, loc="left")
    ax.set_yticks([])
for ax in axes.flat[len(top):]:
    ax.set_visible(False)
fig.suptitle(f"Strongest separating features (K={K0}, densities)", x=0.01, ha="left", fontweight="bold")
fig.legend(*axes.flat[0].get_legend_handles_labels(), loc="upper right", ncols=2)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_figure(fig, cfg, "feat_top_distributions")
plt.show()

## 4. Categorical features

Later-purchase rate per value (train split, default K). The dashed line is the base
rate; values with fewer than 30 sessions are hidden. `main_category` shows the top values
the preprocessor keeps (`top_n_categories`).

In [ ]:
t = train[K0]
base = t[TARGET].mean()
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
cat_tables = {}
for ax, f in zip(axes.flat, CATEGORICAL):
    d = (t.group_by(f).agg(sessions=pl.len(), purchase_rate=pl.col(TARGET).mean())
         .filter(pl.col("sessions") >= 30))
    d = d.sort(f) if f in ("hour_of_day", "day_of_week") else d.sort("sessions", descending=True).head(
        cfg["features"]["top_n_categories"] if f == "main_category" else 20)
    cat_tables[f] = d
    labels = [str(v) for v in d[f].to_list()]
    if f in ("main_category", "last_event_type"):
        ax.barh(labels[::-1], d["purchase_rate"].to_list()[::-1], height=0.6, color=PALETTE[0])
        ax.axvline(base, color=INK_MUTED, linestyle="--", linewidth=1)
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(axis="y", visible=False)
        ax.tick_params(axis="y", labelsize=8 if len(labels) > 10 else 10)
    else:
        ax.bar(labels, d["purchase_rate"], width=0.7, color=PALETTE[0])
        ax.axhline(base, color=INK_MUTED, linestyle="--", linewidth=1)
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(axis="x", visible=False)
        ax.tick_params(axis="x", labelsize=8)
    ax.set_title(f"{f}  [{GROUP_OF[f]}]", fontsize=10)
fig.suptitle(f"Later-purchase rate by categorical value (K={K0}, dashed = base rate {base:.2%})",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, cfg, "feat_categorical")
plt.show()
display(cat_tables["last_event_type"])

## 5. Single-feature predictive power

Each numeric feature used **alone** as a score (higher value = more likely to buy, or
the reverse — whichever direction is better, so ROC-AUC ≥ 0.5). PR-AUC is compared
with the base rate (lift). This is a ceiling-free, model-free reference: anything the
models gain beyond the best single feature comes from combining features.

In [ ]:
rows = []
for k in KS:
    t = train[k]
    y = t[TARGET].to_numpy()
    if y.min() == y.max():
        continue
    for f in NUMERIC:
        x = t[f].cast(pl.Float64).to_numpy()
        auc = roc_auc_score(y, x)
        direction = 1 if auc >= 0.5 else -1
        rows.append({"K": k, "feature": f, "group": GROUP_OF[f], "direction": "+" if direction > 0 else "−",
                     "roc_auc": max(auc, 1 - auc), "pr_auc": average_precision_score(y, direction * x),
                     "base_rate": y.mean()})
uni = pl.DataFrame(rows).with_columns(pr_lift=pl.col("pr_auc") / pl.col("base_rate"))
uni_k0 = uni.filter(pl.col("K") == K0).sort("roc_auc", descending=True)
display(uni_k0)
summary["best_single_feature"] = uni_k0.head(3).select("feature", "roc_auc", "pr_lift").to_dicts()

fig, ax = plt.subplots(figsize=(8, 0.32 * uni_k0.height + 1.2))
d = uni_k0.reverse()
ax.barh(d["feature"], d["roc_auc"] - 0.5, left=0.5, height=0.6, color=[GROUP_COLOURS[g] for g in d["group"]])
ax.axvline(0.5, color=INK_MUTED, linewidth=1)
ax.set_xlim(0.45, min(1.0, d["roc_auc"].max() + 0.08))
ax.grid(axis="y", visible=False)
ax.set_xlabel("ROC-AUC of the feature alone (0.5 = no signal)")
ax.set_title(f"Single-feature ROC-AUC (train, K={K0})")
group_legend(ax, loc="lower right")
save_figure(fig, cfg, "feat_univariate_auc")
plt.show()

## 6. Redundancy

Spearman rank correlation between numeric features (train, default K). Pairs above
|ρ| = 0.8 are listed: they carry largely the same information, which matters for
reading linear-model coefficients and SHAP values (credit gets split between them).
Some pairs are identities, not data findings: with K fixed, `price_sum` = K × `price_mean`,
`view_count` = K − `cart_count` (no remove events in this data), and `has_cart` ⇔ `first_cart_position` ≤ K.

In [ ]:
t = train[K0]
ranked = np.column_stack([t[f].cast(pl.Float64).rank(method="average").to_numpy() for f in NUMERIC])
with np.errstate(invalid="ignore", divide="ignore"):
    rho = np.corrcoef(ranked, rowvar=False)
rho = np.nan_to_num(rho)  # constant columns (e.g. on tiny synthetic data) -> 0

cmap = LinearSegmentedColormap.from_list("div", [PALETTE[1], "#f0efec", PALETTE[0]])
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(rho, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC)), NUMERIC, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(NUMERIC)), NUMERIC, fontsize=8)
for tick, f in zip(ax.get_yticklabels(), NUMERIC):
    tick.set_color(GROUP_COLOURS[GROUP_OF[f]])
for tick, f in zip(ax.get_xticklabels(), NUMERIC):
    tick.set_color(GROUP_COLOURS[GROUP_OF[f]])
ax.grid(False)
for i in range(len(NUMERIC)):
    for j in range(len(NUMERIC)):
        if i != j and abs(rho[i, j]) >= 0.8:
            ax.text(j, i, f"{rho[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white")
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman ρ")
ax.set_title(f"Feature rank correlation (train, K={K0}; labels coloured by group)")
save_figure(fig, cfg, "feat_correlation")
plt.show()

pairs = pl.DataFrame([
    {"a": NUMERIC[i], "b": NUMERIC[j], "rho": float(rho[i, j]), "groups": f"{GROUP_OF[NUMERIC[i]]} / {GROUP_OF[NUMERIC[j]]}"}
    for i in range(len(NUMERIC)) for j in range(i + 1, len(NUMERIC)) if abs(rho[i, j]) >= 0.8
], schema={"a": pl.String, "b": pl.String, "rho": pl.Float64, "groups": pl.String}).sort(pl.col("rho").abs(), descending=True)
display(pairs)
summary["high_corr_pairs"] = pairs.to_dicts()

**Why H3 is hard by construction.** With K fixed, the ratios are exact functions of the
G2 counts: the same count combination always gives the same ratios. (With no remove events,
G3 is the single ratio `cart_to_view_ratio` = cart ÷ (K − cart + ε).) So G3 can
add nothing to a model that already has G2 *and* can learn any function of it. Any gain
in the ablation comes from making that function easier to learn — which is why the
exploratory variant asks the opposite question: can the ratios **replace** the cart block?

In [ ]:
rows = []
for k in KS:
    t = tables[k]
    combos = t.group_by(COUNTS).agg(sessions=pl.len(), *[pl.col(r).n_unique().alias(r) for r in RATIOS])
    worst = int(max(combos[r].max() for r in RATIOS)) if RATIOS else 1
    rows.append({"K": k, "count_combinations": combos.height,
                 "max_distinct_ratio_values_per_combination": worst,
                 "ratios_determined_by_counts": worst == 1})
determinism = pl.DataFrame(rows)
display(determinism)
summary["ratios_determined_by_counts"] = bool(determinism["ratios_determined_by_counts"].all())

## 7. Heavy tails and the log transform

A ratio's denominator can be zero (e.g. no views in an all-cart prefix), giving values
near count ÷ ε = 10⁶. Trees only use the order, so they
are unaffected; linear and neural models get `log1p` and then standard scaling.

In [ ]:
t = train[K0]
eps_hit = t.select(
    cart_to_view_zero_views=(pl.col("view_count") == 0).mean(),
    ratio_above_1000=pl.any_horizontal([pl.col(r) > 1000 for r in RATIOS]).mean() if RATIOS else pl.lit(0.0),
)
display(eps_hit)

tail = [f for f in ("session_duration_sec", "cart_to_view_ratio", "price_sum", "price_std") if f in NUMERIC]
fig, axes = plt.subplots(2, len(tail), figsize=(3.2 * len(tail), 5))
for j, f in enumerate(tail):
    raw = t[f].cast(pl.Float64).to_numpy()
    axes[0, j].hist(raw, bins=40, color=PALETTE[0])
    axes[1, j].hist(np.log1p(np.clip(raw, 0, None)), bins=40, color=PALETTE[2])
    axes[0, j].set_title(f, fontsize=10)
    for ax in axes[:, j]:
        ax.set_yticks([])
        ax.grid(axis="x", visible=False)
axes[0, 0].set_ylabel("raw")
axes[1, 0].set_ylabel("log1p")
fig.suptitle(f"Raw vs log1p scale (train, K={K0})", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, cfg, "feat_log_transform")
plt.show()

## 8. Drift across the chronological split

The preprocessor is fitted on train; after scaling, a feature's mean on validation or
test is its shift **in training standard deviations** (0 = no drift). Shifts beyond
±0.1 SD are worth naming in the threats-to-validity section. The base rate is shown too.

In [ ]:
splits, pre = prepare_splits(cfg, K0, "logistic_regression")
cols = splits["train"].columns
numeric_cols = [c for c in cols if c in NUMERIC]
idx = [cols.index(c) for c in numeric_cols]
drift = pl.DataFrame({
    "feature": numeric_cols,
    "group": [GROUP_OF[c] for c in numeric_cols],
    "val_shift_sd": splits["val"].X[:, idx].mean(axis=0).astype(float),
    "test_shift_sd": splits["test"].X[:, idx].mean(axis=0).astype(float),
}).sort(pl.col("test_shift_sd").abs(), descending=True)
display(drift)
print("base rate  train {:.3%}  val {:.3%}  test {:.3%}".format(*(splits[s].y.mean() for s in ("train", "val", "test"))))
print(f"model columns after one-hot encoding (LR / MLP): {len(cols)}")
summary["max_test_shift_sd"] = drift.head(3).select("feature", "test_shift_sd").to_dicts()

d = drift.sort(pl.col("test_shift_sd").abs())
fig, ax = plt.subplots(figsize=(8, 0.32 * d.height + 1.2))
ax.axvspan(-0.1, 0.1, color="#f0efec", zorder=0)
ax.axvline(0, color=INK_MUTED, linewidth=1)
ax.scatter(d["val_shift_sd"], range(d.height), s=40, color=PALETTE[0], label="validation", zorder=3,
           edgecolor="white", linewidth=1.2)
ax.scatter(d["test_shift_sd"], range(d.height), s=40, color=PALETTE[1], marker="D", label="test", zorder=3,
           edgecolor="white", linewidth=1.2)
lim = max(0.15, 1.15 * float(np.abs(np.r_[d["val_shift_sd"].to_numpy(), d["test_shift_sd"].to_numpy()]).max()))
ax.set_xlim(-lim, lim)
ax.set_yticks(range(d.height), d["feature"])
ax.grid(axis="y", visible=False)
ax.set_xlabel("mean shift from train (in training SDs)")
ax.set_title(f"Feature drift after the split (K={K0}; shaded = ±0.1 SD)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
save_figure(fig, cfg, "feat_drift")
plt.show()

## 9. What the GRU sees

The GRU reads the K events one by one instead of aggregates. If the *order* of events
matters (H1c), buyers and non-buyers should differ by position, not just by count:
the share of cart events at each position, train split.

In [ ]:
fig, axes = plt.subplots(1, len(KS), figsize=(4 * len(KS), 3.6), sharey=True)
axes = np.atleast_1d(axes)
pos_rows = []
for ax, k in zip(axes, KS):
    labels = train[k].select("user_session", TARGET)
    pre_ev = (pl.scan_parquet(processed_dir_for_k(cfg, k) / "prefix_events.parquet")
              .join(labels.lazy(), on="user_session", how="inner")
              .group_by("pos", TARGET).agg(cart_share=(pl.col("event_type") == "cart").mean())
              .sort("pos").collect())
    pos_rows.append(pre_ev.with_columns(K=pl.lit(k)))
    for cls, label in ((0, "non-buyer"), (1, "buyer")):
        d = pre_ev.filter(pl.col(TARGET) == cls)
        ax.plot(d["pos"], d["cart_share"], marker="o", markersize=4, color=TARGET_COLOURS[cls], label=label)
    ax.set_title(f"K={k}", fontsize=10)
    ax.set_xlabel("position in prefix")
    ax.set_xticks(range(1, k + 1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].set_ylabel("share of events that are 'cart'")
axes[-1].legend()
fig.suptitle("Cart events by position, buyers vs non-buyers (train)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, cfg, "feat_cart_by_position")
plt.show()

# Does a late cart matter more than an early one? Purchase rate by first cart position.
fcp = (train[K0].group_by("first_cart_position").agg(sessions=pl.len(), purchase_rate=pl.col(TARGET).mean())
       .sort("first_cart_position"))
display(fcp)

## 10. Key numbers

In [ ]:
lines = [
    f"Integrity checks: {'all passed' if summary['integrity_ok'] else 'FAILED'} for K={KS}.",
    "Largest buyer/non-buyer differences (K={}): ".format(K0) + ", ".join(
        f"{r['feature']} ({r['smd']:+.2f})" for r in summary["top_smd"][:3]) + ".",
    "Best single features (ROC-AUC): " + ", ".join(
        f"{r['feature']} {r['roc_auc']:.3f} (PR lift {r['pr_lift']:.1f}×)" for r in summary["best_single_feature"]) + ".",
    f"{len(summary['high_corr_pairs'])} numeric feature pairs with |Spearman ρ| ≥ 0.8.",
    "Ratios fully determined by the G2 counts: " + ("yes" if summary["ratios_determined_by_counts"] else "no") + ".",
    "Largest train→test shifts: " + ", ".join(
        f"{r['feature']} {r['test_shift_sd']:+.2f} SD" for r in summary["max_test_shift_sd"]) + ".",
]
print("\n".join(f"- {line}" for line in lines))

out = resolve_path(cfg, "tables_dir") / f"feature_analysis_{MODE}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2, default=float))
print(f"\nsaved {out}")